In [41]:
import sys
sys.path.append('/Users/hamza/Desktop/testMax/ai8x-training/')  # Add the path where ai8x is located
sys.path.append('/Users/hamza/Desktop/testMax/ai8x-training/')  # Add the path where ai8x_blocks is located

In [42]:
from torch.utils.data import TensorDataset, DataLoader
from torchinfo import summary  # For model summary
from sklearn.model_selection import train_test_split
from sklearn.metrics import precision_score, recall_score, f1_score
import torch.optim as optim

In [43]:
from torch import nn

import ai8x
import ai8x_blocks

from torchinfo import summary  # Importing torchinfo for model summary

In [44]:
from ai8x import (
    FusedConv1dReLU,
    FusedMaxPoolConv1dReLU,
    FusedLinearReLU,
    Add,
    # Add other ai8x modules if needed
)

In [45]:
# Initialize the ai8x device
from ai8x import set_device

set_device(
    device=87,          # Use 85 for MAX78000 or 87 for MAX78002
    simulate=True,      # Set to False when deploying to actual hardware
    round_avg=True,     # Hardware rounding mode for AvgPool
    verbose=True        # Enable verbose logging
)

Configuring device: MAX78002, simulate=True.


### Model with ReLU activation

In [46]:
import scipy.io
import random
import numpy as np

In [72]:
# Set random seeds
random.seed(42)           # Python random module
np.random.seed(42)        # NumPy random
torch.manual_seed(42)     # PyTorch CPU random

In [47]:
# Load the .mat file
CTF_class_mov = scipy.io.loadmat('Datasets/CTF_Class_mov_final.mat')
CTF_class_static = scipy.io.loadmat('Datasets/CTF_Class_static_final.mat')

CTF_corridor_mov = scipy.io.loadmat('Datasets/CTF_Corridor_mov_final.mat')
CTF_corridor_static = scipy.io.loadmat('Datasets/CTF_Corridor_static_final.mat')

CTF_lab_mov = scipy.io.loadmat('Datasets/CTF_Lab_mov_final.mat')
CTF_lab_static = scipy.io.loadmat('Datasets/CTF_Lab_static_final.mat')

CTF_SC_mov = scipy.io.loadmat('Datasets/CTF_SC_mov_final.mat')
CTF_SC_static = scipy.io.loadmat('Datasets/CTF_SC_static_final.mat')

In [48]:
# Access the 4th item in each loaded variable
#static
CTF_class_static_array = CTF_class_static[list(CTF_class_static.keys())[3]].T
CTF_corridor_static_array = CTF_corridor_static[list(CTF_corridor_static.keys())[3]].T
CTF_lab_static_array = CTF_lab_static[list(CTF_lab_static.keys())[3]].T
CTF_SC_static_array = CTF_SC_static[list(CTF_SC_static.keys())[3]].T

# mov
CTF_class_mov_array = CTF_class_mov[list(CTF_class_mov.keys())[3]].T
CTF_corridor_mov_array = CTF_corridor_mov[list(CTF_corridor_mov.keys())[3]].T
CTF_lab_mov_array = CTF_lab_mov[list(CTF_lab_mov.keys())[3]].T
CTF_SC_mov_array = CTF_SC_mov[list(CTF_SC_mov.keys())[3]].T

CTF_class_static_array.shape

(38800, 2, 101)

In [49]:
# Combine real and imaginary parts for the 4th item in each loaded variable using stack
CTF_class = np.concatenate((CTF_class_static_array, CTF_class_mov_array), axis=0)
CTF_corridor = np.concatenate((CTF_corridor_static_array, CTF_corridor_mov_array), axis=0)
CTF_lab = np.concatenate((CTF_lab_static_array, CTF_lab_mov_array), axis=0)
CTF_SC = np.concatenate((CTF_SC_static_array, CTF_SC_mov_array), axis=0)

In [50]:
CTF_SC.shape

(77600, 2, 101)

In [51]:
# Combine 145 training points randomly selected from each env dataset into a single training set array, 
# and combine 49 testing points randomly selected from each dataset into a single testing set array.

# Number of unique grid points
num_grid_points = 194*2 #388

# Number of measurements per grid point
num_measurements = 200

# Number of grid points to select for training
num_train_points = int(0.75 * num_grid_points)  # 291

# Initialize empty lists for training and test sets
train_set = []
test_set = []

# For each array (representing a different environment)
for array in [CTF_class, CTF_corridor, CTF_lab, CTF_SC]:
    # Reshape the array to separate the measurements for each grid point
    reshaped_array = array.reshape(num_grid_points, num_measurements, -1, 2)
    
    # Randomly select grid points for training
    train_points = random.sample(range(num_grid_points), num_train_points)
    #print(len(train_points))
        
    # Get boolean array for test points
    test_points_bool = ~np.isin(range(num_grid_points), train_points)

    # Print test points
    # test_points = np.arange(num_grid_points)[test_points_bool]
    # print("Test points len:", len(test_points.tolist()))
    
    # Add the selected grid points to the training set and the rest to the test set
    train_set.append(reshaped_array[train_points])
    test_set.append(reshaped_array[test_points_bool])
    
    # ~ is the logical NOT operator, so ~np.isin(shuffle_index, train_points) gives all the indices not in train_points.
# Concatenate the data from all environments
train_set = np.concatenate(train_set, axis=0)
test_set = np.concatenate(test_set, axis=0)

print("Training set:",train_set.shape)
print("Testing set:",test_set.shape)

Training set: (1164, 200, 101, 2)
Testing set: (388, 200, 101, 2)


In [52]:
# Reshaping the arrays
large_X_train = train_set.reshape([-1,101,2])
large_X_test = test_set.reshape([-1,101,2])
print("large_X_train after reshape:", large_X_train.shape)
print("large_X_test after reshape:", large_X_test.shape)

large_X_train after reshape: (232800, 101, 2)
large_X_test after reshape: (77600, 101, 2)


In [53]:
# Create Labels for training data
#static
label1 = np.ones([291*200]) * 0  # class
label2 = np.ones([291*200]) * 1  # corridor
label3 = np.ones([291*200]) * 2  # lab
label4 = np.ones([291*200]) * 3  # SC

large_Y_train = np.concatenate([label1, label2, label3, label4])

label5 = np.ones([97*200]) * 0  # class
label6 = np.ones([97*200]) * 1  # corridor
label7 = np.ones([97*200]) * 2  # lab
label8 = np.ones([97*200]) * 3  # SC

large_Y_test = np.concatenate([label5, label6, label7, label8])

print("Training labels:", large_Y_train.shape)
print("Testing labels:", large_Y_test.shape)

Training labels: (232800,)
Testing labels: (77600,)


In [54]:
# Shuffle Data
shuffle_index1 = random.sample(range(0,232800), 232800)
#print(shuffle_index1)

# Shuffle data using the shuffled indices
large_X_train_new = large_X_train[shuffle_index1, :, :]
print(shuffle_index1[232797:])

large_Y_train = large_Y_train[shuffle_index1]
large_Y_train

[42068, 144446, 119953]


array([0., 3., 0., ..., 0., 2., 2.])

In [55]:
# Shuffle testing data
shuffle_index2 = random.sample(range(0,77600), 77600)

# Shuffle testing data using the shuffled indices
large_X_test_new = large_X_test[shuffle_index2, :, :]
#print(shuffle_index2[0:3])
print(shuffle_index2[77597:])

large_Y_test = large_Y_test[shuffle_index2]
large_Y_test

[36092, 23243, 15570]


array([3., 2., 0., ..., 1., 1., 0.])

In [56]:
import torch
import torch.nn.functional as F

# Converting labels to categorical one-hot encoding
y_train = F.one_hot(torch.tensor(large_Y_train, dtype=torch.long), num_classes=4).float()  # (232000, 4)
y_test = F.one_hot(torch.tensor(large_Y_test, dtype=torch.long), num_classes=4).float()   # (78400, 4)

# Convert to PyTorch tensors
X_train = torch.tensor(large_X_train_new, dtype=torch.float32)  # (232000, 101, 2)
X_test = torch.tensor(large_X_test_new, dtype=torch.float32)    # (77600, 101, 2)

In [57]:
print("Training labels:", y_train.shape)
print("Testing labels:", y_test.shape)

print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

Training labels: torch.Size([232800, 4])
Testing labels: torch.Size([77600, 4])
Training data: torch.Size([232800, 101, 2])
Testing data: torch.Size([77600, 101, 2])


In [109]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
import torch.optim as optim

# Define the PyTorch model equivalent to the TensorFlow model
class CNNModel(nn.Module):
    def __init__(self):
        super(CNNModel, self).__init__()
        
        # First Conv2D layer: 32 filters, kernel (3,1), input (101, 2, 1)
        # PyTorch Conv2d expects (batch, channels, height, width)
        # So we need to reshape from (batch, 101, 2) to (batch, 1, 101, 2)
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=5, kernel_size=(3, 3), stride=(1, 1), padding=1)
        self.bn1 = nn.BatchNorm2d(5)
        
        # Second Conv2D layer: 32 filters, kernel (3,1)
        self.conv2 = nn.Conv2d(in_channels=5, out_channels=5, kernel_size=(3, 3), stride=(1, 1), padding=1)
        self.bn2 = nn.BatchNorm2d(5)
        
        # Calculate flattened size after conv layers
        # After conv1: (101-3+1, 2-1+1) = (99, 2)
        # After conv2: (99-3+1, 2-1+1) = (97, 2)
        # Flattened: 32 * 97 * 2 = 6208
        self.fc1 = nn.Linear(5 * 101 * 2, 100)
        self.dropout = nn.Dropout(0.4)
        self.fc2 = nn.Linear(100, 4)
        
    def forward(self, x):
        
        # print(f"Input shape: {x.shape}")
        # Reshape input from (batch, 101, 2) to (batch, 1, 101, 2)
        x = x.unsqueeze(1)  # Add channel dimension

        # print(f"After unsqueeze: {x.shape}")
        
        # First conv block
        x = self.conv1(x)
        # print(f"After conv1: {x.shape}")
        x = self.bn1(x)
        x = F.relu(x)
        # print(f"After bn1+relu: {x.shape}")
        
        # Second conv block
        x = self.conv2(x)
        # print(f"After conv2: {x.shape}")
        x = self.bn2(x)
        x = F.relu(x)
        # print(f"After bn2+relu: {x.shape}")
        
        # Flatten
        x = x.view(x.size(0), -1)
        # print(f"After flatten: {x.shape}")
        
        # Fully connected layers
        x = self.fc1(x)
        # print(f"After fc1: {x.shape}")
        x = F.relu(x)
        x = self.dropout(x)
        x = self.fc2(x)
        # print(f"Final output: {x.shape}")
        
        return x
        # Then your criterion = nn.CrossEntropyLoss() will internally apply log_softmax + nll_loss.

# Initialize the model
model = CNNModel()
print(model)

CNNModel(
  (conv1): Conv2d(1, 5, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn1): BatchNorm2d(5, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (conv2): Conv2d(5, 5, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (bn2): BatchNorm2d(5, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (fc1): Linear(in_features=1010, out_features=100, bias=True)
  (dropout): Dropout(p=0.4, inplace=False)
  (fc2): Linear(in_features=100, out_features=4, bias=True)
)


In [110]:
# Quick shape verification
model = CNNModel()
dummy_input = torch.randn(1, 101, 2)  # Single sample

print("=== SHAPE VERIFICATION ===")
print(f"Input to model: {dummy_input.shape}")

# Run one forward pass to see shapes
with torch.no_grad():
    output = model(dummy_input)
    
print(f"Output from model: {output.shape}")
print("Expected: torch.Size([1, 4])")  # 1 sample, 

=== SHAPE VERIFICATION ===
Input to model: torch.Size([1, 101, 2])
Output from model: torch.Size([1, 4])
Expected: torch.Size([1, 4])


In [111]:
# Define learning rate scheduler function
def lr_schedule(epoch):
    lr = 0.001
    if epoch > 2:
        lr *= 0.5e-3
    elif epoch > 4:
        lr *= 1e-3
    return lr

# Set up optimizer and loss function
optimizer = optim.Adam(model.parameters(), lr=lr_schedule(0))
criterion = nn.CrossEntropyLoss()

# Create data loaders
batch_size = 128
train_dataset = TensorDataset(X_train, y_train)
test_dataset = TensorDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [112]:
def train_model(model, train_loader, test_loader, optimizer, criterion, num_epochs=10):
    train_losses = []
    train_accuracies = []
    val_losses = []
    val_accuracies = []
    
    for epoch in range(num_epochs):
        # Update learning rate
        new_lr = lr_schedule(epoch)
        for param_group in optimizer.param_groups:
            param_group['lr'] = new_lr
        
        print(f'\nEpoch [{epoch+1}/{num_epochs}], Learning Rate: {new_lr:.6f}')
        print('-' * 60)
        
        # Training phase
        model.train()
        train_loss = 0.0
        train_correct = 0
        train_total = 0
        
        for batch_idx, (data, target) in enumerate(train_loader):
            optimizer.zero_grad()
            output = model(data)
            
            # Convert one-hot to class indices for CrossEntropyLoss
            target_indices = torch.argmax(target, dim=1)
            loss = criterion(output, target_indices)
            
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item()
            _, predicted = torch.max(output.data, 1)
            train_total += target.size(0)
            train_correct += (predicted == target_indices).sum().item()
            
            # Print progress every 1000 batches
            if batch_idx % 1000 == 0:
                current_acc = 100. * train_correct / train_total if train_total > 0 else 0
                print(f'  Batch [{batch_idx:4d}/{len(train_loader)}] | '
                      f'Loss: {loss.item():.4f} | '
                      f'Acc: {current_acc:.2f}% | '
                      f'Processed: {train_total}/{len(train_loader.dataset)}')
        
        # Validation phase
        model.eval()
        val_loss = 0.0
        val_correct = 0
        val_total = 0
        
        print(f'  Running validation...')
        with torch.no_grad():
            for batch_idx, (data, target) in enumerate(test_loader):
                output = model(data)
                target_indices = torch.argmax(target, dim=1)
                loss = criterion(output, target_indices)
                
                val_loss += loss.item()
                _, predicted = torch.max(output.data, 1)
                val_total += target.size(0)
                val_correct += (predicted == target_indices).sum().item()
                
                # Print validation progress every 500 batches
                if batch_idx % 500 == 0:
                    current_val_acc = 100. * val_correct / val_total if val_total > 0 else 0
                    print(f'    Val Batch [{batch_idx:3d}/{len(test_loader)}] | '
                          f'Loss: {loss.item():.4f} | '
                          f'Acc: {current_val_acc:.2f}%')
        
        # Calculate averages
        train_loss /= len(train_loader)
        val_loss /= len(test_loader)
        train_acc = 100. * train_correct / train_total
        val_acc = 100. * val_correct / val_total
        
        # Store metrics
        train_losses.append(train_loss)
        train_accuracies.append(train_acc)
        val_losses.append(val_loss)
        val_accuracies.append(val_acc)
        
        print(f'\n  EPOCH {epoch+1} SUMMARY:')
        print(f'  Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%')
        print(f'  Val Loss:   {val_loss:.4f} | Val Acc:   {val_acc:.2f}%')
        print('=' * 60)
    
    return train_losses, train_accuracies, val_losses, val_accuracies

# Train the model with verbose output
history = train_model(model, train_loader, test_loader, optimizer, criterion, num_epochs=10)


Epoch [1/10], Learning Rate: 0.001000
------------------------------------------------------------
  Batch [   0/1819] | Loss: 1.4174 | Acc: 25.00% | Processed: 128/232800
  Batch [1000/1819] | Loss: 0.0016 | Acc: 98.02% | Processed: 128128/232800
  Running validation...
    Val Batch [  0/607] | Loss: 10.6029 | Acc: 32.03%
    Val Batch [500/607] | Loss: 8.1834 | Acc: 35.56%

  EPOCH 1 SUMMARY:
  Train Loss: 0.0345 | Train Acc: 98.86%
  Val Loss:   9.7414 | Val Acc:   35.56%

Epoch [2/10], Learning Rate: 0.001000
------------------------------------------------------------
  Batch [   0/1819] | Loss: 0.0036 | Acc: 100.00% | Processed: 128/232800
  Batch [1000/1819] | Loss: 0.0001 | Acc: 99.92% | Processed: 128128/232800
  Running validation...
    Val Batch [  0/607] | Loss: 187.0561 | Acc: 26.56%
    Val Batch [500/607] | Loss: 214.4958 | Acc: 24.90%

  EPOCH 2 SUMMARY:
  Train Loss: 0.0026 | Train Acc: 99.92%
  Val Loss:   190.7608 | Val Acc:   25.00%

Epoch [3/10], Learning Rate: 

## Define Model Using ai8x Blocks for MAX78000/2

In [86]:
# Initialize the ai8x device
from ai8x import set_device

set_device(
    device=87,          # Use 85 for MAX78000 or 87 for MAX78002
    simulate=True,      # Set to False when deploying to actual hardware
    round_avg=True,     # Hardware rounding mode for AvgPool
    verbose=True        # Enable verbose logging
)

Configuring device: MAX78002, simulate=True.


In [94]:
# Define the AI8X-adapted model for MAX78000/MAX78002
class AI8X_CNNModel(nn.Module):
    def __init__(self):
        super(AI8X_CNNModel, self).__init__()
        
        # First Conv2D layer with BatchNorm and ReLU fused: 32 filters, kernel (3,1)
        # Input shape: (batch, 1, 101, 2) - we'll add the channel dimension in forward()
        self.conv1 = ai8x.FusedConv2dBNReLU(
            in_channels=1, 
            out_channels=5, 
            kernel_size=(3, 3), 
            stride=1, 
            padding=1,
            bias=True
        )
        
        # Second Conv2D layer with BatchNorm and ReLU fused: 32 filters, kernel (3,1)
        self.conv2 = ai8x.FusedConv2dBNReLU(
            in_channels=5, 
            out_channels=5, 
            kernel_size=(3, 3), 
            stride=1, 
            padding=1,
            bias=True
        )
        
        # Calculate flattened size after conv layers
        # After conv1: (101-3+1, 2-1+1) = (99, 2)
        # After conv2: (99-3+1, 2-1+1) = (97, 2)
        # Flattened: 32 * 97 * 2 = 6208
        
        # First fully connected layer with ReLU
        self.fc1 = ai8x.FusedLinearReLU(
            in_features=5 * 97 * 2, 
            out_features=100,
            bias=True
        )
        
        # Dropout (ai8x doesn't have special dropout, use regular PyTorch)
        self.dropout = nn.Dropout(0.4)
        
        # Final fully connected layer (no activation - we'll use CrossEntropyLoss)
        self.fc2 = ai8x.Linear(
            in_features=100, 
            out_features=4,
            bias=True
        )
        
    def forward(self, x):
        # Reshape input from (batch, 101, 2) to (batch, 1, 101, 2)
        x = x.unsqueeze(1)  # Add channel dimension
        
        # First fused conv block (Conv2d + BatchNorm + ReLU)
        x = self.conv1(x)
        
        # Second fused conv block (Conv2d + BatchNorm + ReLU)
        x = self.conv2(x)
        
        # Flatten
        x = x.view(x.size(0), -1)
        
        # First fully connected layer with ReLU
        x = self.fc1(x)
        
        # Dropout
        x = self.dropout(x)
        
        # Final fully connected layer
        x = self.fc2(x)
        
        return x  # Return logits for CrossEntropyLoss


In [95]:
# Initialize the AI8X model
model = AI8X_CNNModel()
print("AI8X Model Architecture:")
print(model)

AI8X Model Architecture:
AI8X_CNNModel(
  (conv1): FusedConv2dBNReLU(
    (activate): ReLU(inplace=True)
    (op): Conv2d(1, 5, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (bn): BatchNorm2d(5, eps=1e-05, momentum=0.05, affine=True, track_running_stats=True)
    (calc_out_shift): OutputShiftPassthrough()
    (calc_weight_scale): One()
    (scale): Scaler()
    (calc_out_scale): OutputScale()
    (quantize_weight): Quantize()
    (quantize_bias): Quantize()
    (clamp_weight): Empty()
    (clamp_bias): Empty()
    (quantize): Quantize()
    (clamp): Clamp()
    (quantize_pool): Empty()
    (clamp_pool): Empty()
  )
  (conv2): FusedConv2dBNReLU(
    (activate): ReLU(inplace=True)
    (op): Conv2d(5, 5, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (bn): BatchNorm2d(5, eps=1e-05, momentum=0.05, affine=True, track_running_stats=True)
    (calc_out_shift): OutputShiftPassthrough()
    (calc_weight_scale): One()
    (scale): Scaler()
    (calc_out_scale): OutputScale()
   